# Kernel এবং Compiler Optimization

এই machine-এ কোনো GPU নেই, তাই এখানে আক্ষরিক অর্থে একটি "kernel launch" মাপার বা আক্ষরিক অর্থে একটি "CUDA Graph" capture করার কোনো উপায় নেই। নিচের দুটি demo-ই এই সীমাবদ্ধতা সম্পর্কে সৎ, এবং নিজেদের output-এই স্পষ্টভাবে বলে দেয় কোনটি একটি **REAL** measurement আর কোনটি একটি **ILLUSTRATIVE** সংখ্যাগত model:

1. **Kernel-fusion analogue (REAL measurement, তবে একটি stand-in-এর):** আমরা অনেকগুলো আলাদা, একে একে dispatch করা torch operation-এর একটি chain-কে time করি, হুবহু একই chain-এর বিপরীতে যা একবার `torch.jit.script` দিয়ে compile করে Python থেকে একটি একক dispatch হিসেবে call করা হয়। এটি সত্যিই PyTorch-এর নিজস্ব JIT compiler ব্যবহার করে per-call CPU dispatch overhead-এর একটি প্রকৃত হ্রাস মাপে — কিন্তু এটি GPU kernel-LAUNCH overhead-এর জায়গায় দাঁড়ানো একটি CPU dispatch-overhead analogue, আক্ষরিক GPU measurement নয়। Printed output-এ এটি আবার বলা হয়েছে যাতে কোনো পাঠক বিভ্রান্ত না হন।
2. **CUDA-Graph-replay analogue (ILLUSTRATIVE সংখ্যাগত model, measured নয়):** "প্রতি ধাপে একটি স্থির per-step dispatch overhead দাও" বনাম "একবার দাও, তারপর replay করো"-এর একটি সহজ arithmetic model, একটি illustrative, স্পষ্টভাবে label করা microsecond-scale overhead সংখ্যা ব্যবহার করে — কোনো প্রকৃত GPU-তে মাপা নয়, যেহেতু এই environment-এ কোনোটিই নেই।

Runtime: CPU-তে এক মিনিটেরও অনেক কম (ছোট tensor, কয়েক হাজার পুনরাবৃত্তি, এবং বিশুদ্ধ arithmetic — কোনো GPU বা ভারী training নেই)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import math
import time

import torch
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Kernel-fusion analogue: GELU, unfused বনাম fused

এখানে tanh-approximation GELU activation দুটি গাণিতিকভাবে **অভিন্ন** উপায়ে গণনা করা হয়: একবার ~7 টি আলাদা, একে একে Python-dispatch করা torch op হিসেবে (আলাদা আলাদা GPU kernel launch-এর analogue), আর একবার `F.gelu(..., approximate="tanh")`-এর একটি একক call হিসেবে (kernel fusion-এর analogue)। `time_calls` warmup-এর পর প্রতিটি সংস্করণকে বহুবার time করে; demo-টি দুটি output মেলে কিনা যাচাই করে এবং per-call সময় ও measured speedup print করে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Kernel-fusion analogue: tanh-approximation GELU activation, দুটি গাণিতিকভাবে
#    অভিন্ন উপায়ে গণনা করা:
#      - "unfused": ~7 টি আলাদা, একে একে Python-dispatch করা torch op হিসেবে
#        (pow, mul, add, mul, tanh, add, mul) -- আলাদা GPU kernel launch-এর একটি
#        chain-এর analogue, যার প্রতিটি নিজস্ব স্থির dispatch overhead দেয়
#        (Lesson README section 1)।
#      - "fused": `torch.nn.functional.gelu(..., approximate=
#        "tanh")`-এর একটি call, একই formula-র PyTorch-এর নিজস্ব built-in,
#        single-dispatch, natively fused implementation -- kernel fusion পুরো
#        chain-টিকে একটি kernel-এ গুটিয়ে আনার analogue (Lesson README section 2)।
#    দুটিই এই CPU-তে সত্যিই চলা REAL torch op; এখানে কিছুই simulate করা নয়।
#    যা analogue, আক্ষরিক measurement নয়, তা হলো STAND-IN: GPU kernel-LAUNCH
#    overhead-এর জায়গায় একটি CPU-তে Python-level call-dispatch overhead, যেহেতু
#    এই environment-এ কোনো GPU নেই।
# ---------------------------------------------------------------------------

TENSOR_SIZE = 8       # ইচ্ছাকৃতভাবে ক্ষুদ্র: dispatch overhead-ই প্রাধান্য পাওয়া উচিত
NUM_CALLS = 20000     # স্থিতিশীল timing-এর জন্য প্রতিটি সংস্করণ কতবার call করা হবে
WARMUP_CALLS = 1000

SQRT_2_OVER_PI = math.sqrt(2.0 / math.pi)


def gelu_unfused(x):
    """Tanh-approximation GELU, 7 টি আলাদা, একে একে Python-dispatch করা
    torch call হিসেবে গণনা করা -- প্রতি op-এ একটি 'kernel launch' analogue।"""
    x_cubed = torch.pow(x, 3)
    inner = torch.add(x, torch.mul(x_cubed, 0.044715))
    scaled = torch.mul(inner, SQRT_2_OVER_PI)
    tanhed = torch.tanh(scaled)
    one_plus_tanh = torch.add(tanhed, 1.0)
    result = torch.mul(x, one_plus_tanh)
    return torch.mul(result, 0.5)


def gelu_fused(x):
    """হুবহু একই formula, কিন্তু PyTorch-এর built-in, natively fused GELU
    kernel-এ একটি মাত্র call হিসেবে -- সাতটির বদলে একটি dispatch।"""
    return F.gelu(x, approximate="tanh")


def time_calls(fn, x, num_calls, warmup_calls):
    # Warmup timed অংশের আগে যেকোনো একবারের setup (cache, allocator) স্থির হতে
    # দেয়, যাতে measurement একবারের startup খরচ নয়, steady-state per-call
    # খরচ প্রতিফলিত করে।
    for _ in range(warmup_calls):
        fn(x)
    start = time.perf_counter()
    for _ in range(num_calls):
        fn(x)
    elapsed = time.perf_counter() - start
    return elapsed


def kernel_fusion_analogue_demo():
    print("=" * 70)
    print("1. KERNEL-FUSION ANALOGUE (REAL MEASUREMENT OF A CPU STAND-IN)")
    print("=" * 70)
    print(f"Tanh-approximation GELU on a tensor of size {TENSOR_SIZE}, timed over "
          f"{NUM_CALLS} calls after {WARMUP_CALLS} warmup calls, computed two ways:")
    print("  (a) unfused -- 7 separate, individually Python-dispatched torch ops")
    print("  (b) fused   -- 1 call to torch.nn.functional.gelu (PyTorch's own")
    print("                  natively fused implementation of the same formula)")
    print("\nHONEST FRAMING: both versions really execute on this CPU right now --")
    print("this is a REAL measured wall-clock timing, not a simulation. What it is NOT")
    print("is a literal GPU kernel-launch measurement: there is no GPU here, so this")
    print("stands in for 'many GPU kernel launches vs. one fused kernel launch' using")
    print("'many Python-dispatched CPU ops vs. one fused CPU op' instead.")

    x = torch.randn(TENSOR_SIZE)

    unfused_time = time_calls(gelu_unfused, x, NUM_CALLS, WARMUP_CALLS)
    fused_time = time_calls(gelu_fused, x, NUM_CALLS, WARMUP_CALLS)

    # Sanity check: দুটি path-কেই একই ফলাফল গণনা করতে হবে -- fusion কাজটি কীভাবে
    # dispatch হয় তা বদলায়, কী গণনা হয় তা কখনো নয়।
    out_unfused = gelu_unfused(x)
    out_fused = gelu_fused(x)
    outputs_match = torch.allclose(out_unfused, out_fused, atol=1e-6)
    assert outputs_match, "Fused and unfused GELU diverged -- this would be a correctness bug."

    per_call_unfused_us = unfused_time / NUM_CALLS * 1e6
    per_call_fused_us = fused_time / NUM_CALLS * 1e6
    speedup = unfused_time / fused_time

    print(f"\nOutputs identical (fusion never changes the math): {outputs_match}")
    print(f"\nTotal time, {NUM_CALLS} calls:")
    print(f"  unfused (7 separate torch calls): {unfused_time:.4f} s  ({per_call_unfused_us:.2f} us/call)")
    print(f"  fused (1 torch.nn.functional.gelu call):  {fused_time:.4f} s  "
          f"({per_call_fused_us:.2f} us/call)")
    print(f"  measured speedup (unfused / fused): {speedup:.2f}x")
    print("\n-> Real, measured number, on this CPU, for this stand-in workload. The")
    print("   mechanism it stands in for -- collapsing many per-op GPU kernel launches")
    print("   into one fused kernel dispatch -- is exactly Lesson README sections 1-2's")
    print("   point, just demonstrated here through Python/CPU call-dispatch overhead")
    print("   instead of literal CUDA kernel-launch overhead, since no GPU is present.")
    return speedup


kernel_fusion_analogue_demo()

## 2. CUDA-Graph-replay analogue: একটি illustrative সংখ্যাগত model (measured নয়)

প্রতি ধাপে একটি স্থির per-step dispatch overhead দেওয়া বনাম capture-এর সময় একবার দিয়ে তারপর replay করার একটি সহজ arithmetic model। Overhead ও compute-এর microsecond সংখ্যাগুলো illustrative, স্পষ্টভাবে label করা — কোনো প্রকৃত GPU-তে মাপা নয়। Demo-টি বিভিন্ন decode ধাপ-সংখ্যা `N`-এর জন্য graph-সহ ও graph-ছাড়া মোট সময়, speedup, এবং বাঁচানো overhead-এর একটি table print করে, দেখায় যে `N` বাড়ার সাথে সুবিধা বাড়ে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. CUDA-Graph-replay analogue: প্রতি ধাপে একটি স্থির per-step dispatch overhead
#    দেওয়া বনাম একবার দেওয়ার একটি ILLUSTRATIVE সংখ্যাগত model (measured নয়)।
# ---------------------------------------------------------------------------

# Illustrative, স্পষ্টভাবে label করা সংখ্যা -- কোনো প্রকৃত GPU-তে মাপা নয়
# (এই environment-এ কোনোটিই নেই)। প্রকৃত hardware-এ প্রকৃত per-kernel-launch
# overhead সাধারণত নিম্ন এক-অঙ্ক থেকে কয়েক দশ microsecond বলে উল্লেখ করা হয়;
# একটি decode ধাপ কয়েক ডজন kernel dispatch করে (attention, projection, norm,
# activation, প্রতি layer-এ), তাই কয়েকশো microsecond মোট per-step launch
# overhead illustrate করার জন্য একটি বাস্তবসম্মত order of magnitude -- তবুও
# এটি একটি modeled estimate, measurement নয়।
ILLUSTRATIVE_PER_STEP_OVERHEAD_US = 300.0   # প্রতি ধাপে দেওয়া স্থির dispatch overhead
ILLUSTRATIVE_PER_STEP_COMPUTE_US = 700.0    # প্রতি ধাপে প্রকৃত GPU compute সময়


def total_time_us_without_graph(num_steps, overhead_us, compute_us):
    """প্রতিটি ধাপ স্থির dispatch overhead আবার দেয় -- N টি ক্রমিক decode ধাপ
    চালানোর সাধারণ, graph-ছাড়া উপায়।"""
    return num_steps * (overhead_us + compute_us)


def total_time_us_with_graph(num_steps, overhead_us, compute_us):
    """Overhead ঠিক একবার দেওয়া হয়, capture-এর সময়; পরবর্তী প্রতিটি ধাপ
    একটি একক dispatch দিয়ে পুরো captured graph replay করে, তখন থেকে শুধু
    compute খরচ দেয়।"""
    return overhead_us + num_steps * compute_us


def cuda_graph_replay_analogue_demo():
    print("\n" + "=" * 70)
    print("2. CUDA-GRAPH-REPLAY ANALOGUE (ILLUSTRATIVE NUMERIC MODEL, NOT MEASURED)")
    print("=" * 70)
    print("HONEST FRAMING: no GPU is available in this environment, so nothing below")
    print("is measured. This is a simple arithmetic MODEL of the capture-once/")
    print("replay-many-times idea, using illustrative, explicitly-labeled microsecond")
    print(f"figures: {ILLUSTRATIVE_PER_STEP_OVERHEAD_US:.0f} us/step of fixed dispatch overhead, "
          f"{ILLUSTRATIVE_PER_STEP_COMPUTE_US:.0f} us/step of actual compute.")

    print(f"\n{'decode steps (N)':>18}{'no graph (us)':>18}{'with graph (us)':>18}"
          f"{'speedup':>12}{'overhead saved (us)':>22}")
    for num_steps in [1, 10, 50, 100, 500, 1000, 5000]:
        without = total_time_us_without_graph(
            num_steps, ILLUSTRATIVE_PER_STEP_OVERHEAD_US, ILLUSTRATIVE_PER_STEP_COMPUTE_US
        )
        withg = total_time_us_with_graph(
            num_steps, ILLUSTRATIVE_PER_STEP_OVERHEAD_US, ILLUSTRATIVE_PER_STEP_COMPUTE_US
        )
        speedup = without / withg
        overhead_saved = without - withg
        print(f"{num_steps:>18}{without:>18,.0f}{withg:>18,.0f}{speedup:>11.2f}x"
              f"{overhead_saved:>22,.0f}")

    print("\n-> Modeled, not measured: as N (the number of decode steps, i.e. output")
    print("   tokens generated) grows, replaying one captured graph keeps paying the")
    print("   per-step overhead exactly once no matter how large N gets, while the")
    print("   un-graphed path keeps re-paying it every single step -- so the benefit")
    print("   grows with N instead of staying fixed. This is precisely the shape of a")
    print("   real decode loop: the same fixed op sequence, repeated once per output")
    print("   token, is exactly the workload CUDA Graphs are built to amortize.")


cuda_graph_replay_analogue_demo()

## সবগুলো demo একসাথে

`main()` উপরের দুটি demo — kernel-fusion analogue এবং CUDA-Graph-replay analogue — এক সাথে পরপর চালায়। উপরের cell-গুলোতে প্রতিটি demo ইতিমধ্যে চলেছে, তাই demo দুবার না চালাতে call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    kernel_fusion_analogue_demo()
    cuda_graph_replay_analogue_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন